# 🚀 04. Fine-tune YOLO11s — Vietnam Traffic Detection

**Mục tiêu:** Fine-tune mô hình YOLO11s nhận diện phương tiện giao thông (đặc biệt xe máy, người đi bộ) trong bối cảnh camera giao thông Việt Nam.

**Môi trường:** Kaggle Notebook với GPU T4/P100 (30h quota/tuần)

**Dữ liệu:** Dataset đã chuẩn bị sẵn trên Google Drive, upload lên Kaggle Datasets.

### Trước khi chạy notebook này:
1. ✅ Upload dataset `vn_traffic_yolo` lên **Kaggle Datasets** (hoặc chuẩn bị link Google Drive)
2. ✅ Bật **GPU T4 x2** trong Settings → Accelerator
3. ✅ Bật **Internet ON** trong Settings
4. ✅ Add dataset vào notebook (panel phải → + Add data)

---
## 1. Kiểm tra GPU & Cài đặt thư viện

In [ ]:
import torch
import os
import sys

print("=" * 50)
print("KIỂM TRA PHẦN CỨNG")
print("=" * 50)

IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules
print(f"Platform: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   VRAM: {vram:.1f} GB")
    print(f"   PyTorch: {torch.__version__}")
    print(f"   CUDA: {torch.version.cuda}")
    gpu_count = torch.cuda.device_count()
    if gpu_count > 1:
        print(f"   🔥 Phát hiện {gpu_count} GPU → Ultralytics sẽ tự dùng multi-GPU!")
else:
    print("❌ GPU chưa bật!")
    print("👉 Vào Settings (panel phải) → Accelerator → chọn 'GPU T4 x2'")
    raise RuntimeError("Cần GPU để training. Hãy bật GPU rồi restart kernel.")

# Cài Ultralytics
!pip install -q ultralytics

import ultralytics
from ultralytics import YOLO
print(f"\n✅ Ultralytics v{ultralytics.__version__} installed!")

# Kiểm tra nvidia-smi
!nvidia-smi

---
## 2. Chuẩn bị dữ liệu

Chọn **1 trong 2 cách** để nạp dữ liệu:
- **Cách 1 (⭐ Khuyến nghị):** Dataset đã upload lên Kaggle Datasets → load cực nhanh
- **Cách 2:** Tải trực tiếp từ Google Drive bằng `gdown`

In [ ]:
import yaml
import glob
from pathlib import Path

# ╔══════════════════════════════════════════════════════════╗
# ║  CHỌN 1 TRONG 2 CÁCH (comment/uncomment cho phù hợp)  ║
# ╚══════════════════════════════════════════════════════════╝

# --- CÁCH 1: Từ Kaggle Datasets (KHUYẾN NGHỊ) ---
# Trước đó bạn đã upload dataset lên Kaggle Datasets và Add vào notebook
# Sửa tên dataset cho khớp với tên bạn đặt khi upload:
KAGGLE_DATASET_NAME = "vn-traffic-yolo11"  # ← SỬA TÊN DATASET CỦA BẠN
KAGGLE_INPUT = Path("/kaggle/input/datasets/thanhnd2005/vn-traffic-yolo111/vn_traffic")

# Nếu cấu trúc khác (không có subfolder vn_traffic_yolo), thử:
if not KAGGLE_INPUT.exists():
    KAGGLE_INPUT = Path(f"/kaggle/input/{KAGGLE_DATASET_NAME}")

# --- CÁCH 2: Tải từ Google Drive ---
# Uncomment 3 dòng dưới nếu muốn tải từ Drive:
# !pip install -q gdown
# !gdown --folder "https://drive.google.com/drive/folders/FOLDER_ID_CỦA_BẠN" -O /kaggle/working/vn_traffic_yolo --remaining-ok
# KAGGLE_INPUT = Path("/kaggle/working/vn_traffic_yolo")

# ══════════════════════════════════════════════════════════
# THIẾT LẬP ĐƯỜNG DẪN (không cần sửa)
# ══════════════════════════════════════════════════════════
WORK_DIR = Path("/kaggle/working")
DATA_DIR = WORK_DIR / "dataset"
OUTPUT_DIR = WORK_DIR / "runs"

# Kiểm tra dữ liệu nguồn tồn tại
if not KAGGLE_INPUT.exists():
    print(f"❌ Không tìm thấy dataset tại: {KAGGLE_INPUT}")
    print("\n🔍 Các dataset có sẵn trong /kaggle/input/:")
    for d in Path("/kaggle/input").iterdir():
        print(f"   📁 {d.name}")
        for sub in d.iterdir():
            print(f"      └── {sub.name}")
    raise FileNotFoundError("Sửa lại KAGGLE_DATASET_NAME hoặc đường dẫn KAGGLE_INPUT ở trên!")

# Copy data sang /kaggle/working vì /kaggle/input là read-only
if not (DATA_DIR / "images").exists():
    print("📂 Đang copy dữ liệu vào thư mục làm việc...")
    !cp -r {KAGGLE_INPUT} {DATA_DIR}
    print("✅ Copy xong!")
else:
    print("✅ Dữ liệu đã có sẵn trong thư mục làm việc.")

# Tạo data.yaml với đường dẫn Kaggle
data_config = {
    'path': str(DATA_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 6,
    'names': ['car', 'motorcycle', 'truck', 'bus', 'person', 'bicycle']
}

DATA_YAML = WORK_DIR / "data.yaml"
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

# Thống kê dataset
train_imgs = glob.glob(str(DATA_DIR / "images/train/*.[jJ][pP][gG]")) + \
             glob.glob(str(DATA_DIR / "images/train/*.[pP][nN][gG]")) + \
             glob.glob(str(DATA_DIR / "images/train/*.[jJ][pP][eE][gG]"))
val_imgs = glob.glob(str(DATA_DIR / "images/val/*.[jJ][pP][gG]")) + \
           glob.glob(str(DATA_DIR / "images/val/*.[pP][nN][gG]")) + \
           glob.glob(str(DATA_DIR / "images/val/*.[jJ][pP][eE][gG]"))

print(f"\n{'=' * 50}")
print(f"📊 THỐNG KÊ DATASET")
print(f"{'=' * 50}")
print(f"   📄 data.yaml:  {DATA_YAML}")
print(f"   📁 Data dir:   {DATA_DIR}")
print(f"   🏋️ Train:      {len(train_imgs)} ảnh")
print(f"   📝 Val:        {len(val_imgs)} ảnh")
print(f"   🏷️ Classes:    {data_config['names']}")
print(f"{'=' * 50}")

if len(train_imgs) == 0:
    print("\n❌ Không tìm thấy ảnh train!")
    print("Kiểm tra cấu trúc thư mục:")
    !find {DATA_DIR} -type f | head -20
    raise RuntimeError("Dataset rỗng! Kiểm tra lại đường dẫn.")

---
## 3. Kiểm tra Checkpoint cũ (Resume training nếu bị ngắt)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH RESUME TRAINING                               ║
# ╚══════════════════════════════════════════════════════════╝

RESUME_CHECKPOINT = None  # Mặc định: train từ đầu

# --- NẾU BẠN CẦN RESUME (session trước bị ngắt): ---
# Cách A: Upload last.pt lên Kaggle Datasets, add vào notebook, rồi uncomment:
# RESUME_CHECKPOINT = "/kaggle/input/yolo-checkpoint/last.pt"

# Cách B: Tải last.pt từ Google Drive:
# !pip install -q gdown
# !gdown "https://drive.google.com/uc?id=FILE_ID_CỦA_LAST_PT" -O /kaggle/working/last.pt
# RESUME_CHECKPOINT = "/kaggle/working/last.pt"

# Cách C: Nếu last.pt còn trong /kaggle/working từ session trước:
# RESUME_CHECKPOINT = "/kaggle/working/runs/yolo11s_vn_traffic/weights/last.pt"

# ══════════════════════════════════════════════════════════

if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    size_mb = os.path.getsize(RESUME_CHECKPOINT) / (1024 * 1024)
    print(f"🔄 Sẽ RESUME training từ checkpoint:")
    print(f"   📁 {RESUME_CHECKPOINT} ({size_mb:.1f} MB)")
elif RESUME_CHECKPOINT:
    print(f"⚠️ Checkpoint không tồn tại: {RESUME_CHECKPOINT}")
    print("   → Sẽ bắt đầu training MỚI.")
    RESUME_CHECKPOINT = None
else:
    print("🆕 Sẽ bắt đầu training MỚI từ yolo11s.pt pretrained (COCO)")

---
## 4. 🔥 BẮT ĐẦU FINE-TUNE YOLO11s

> ⏱️ **Thời gian ước tính:** 4-8 giờ trên GPU T4 (tùy số lượng ảnh)
>
> 💡 **Chạy ngầm:** Click "Save Version" → "Save & Run All" → Đóng tab an toàn

In [ ]:
from ultralytics import YOLO
import time

start_time = time.time()

if RESUME_CHECKPOINT:
    # ═══ RESUME TRAINING ═══
    print(f"🔄 Resuming training từ: {RESUME_CHECKPOINT}")
    model = YOLO(RESUME_CHECKPOINT)
    results = model.train(resume=True)
else:
    # ═══ TRAIN MỚI TỪ PRETRAINED ═══
    print("🚀 Bắt đầu Fine-tune YOLO11s từ COCO pretrained...")
    print(f"   Dataset: {DATA_YAML}")
    print(f"   Output:  {OUTPUT_DIR}")
    print("=" * 50)
    
    model = YOLO("yolo11s.pt")  # Load pretrained COCO weights

    results = model.train(
        # ── Dataset ──
        data=str(DATA_YAML),
        
        # ── Training config ──
        epochs=80,                  # 60-80 epochs đủ hội tụ
        imgsz=640,                  # Resolution chuẩn YOLO
        batch=16,                   # Phù hợp T4 16GB (giảm xuống 8 nếu OOM)
        patience=15,                # Early stopping: dừng nếu 15 epochs không cải thiện
        
        # ── Optimizer ──
        optimizer="AdamW",          # Tốt cho fine-tuning
        lr0=0.001,                  # LR khởi đầu (nhỏ để giữ pretrained weights)
        lrf=0.01,                   # LR cuối = lr0 × lrf = 0.00001
        warmup_epochs=3,            # Khởi động từ từ 3 epochs đầu
        cos_lr=True,                # Cosine annealing scheduler
        weight_decay=0.0005,        # Regularization
        
        # ── Augmentation ──
        hsv_h=0.015,                # Augment màu sắc
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=0.0,                # Không xoay (camera CCTV cố định)
        translate=0.1,
        scale=0.5,
        flipud=0.0,                 # Không lật dọc
        fliplr=0.5,                 # Lật ngang OK
        mosaic=1.0,                 # Mosaic augmentation
        mixup=0.1,                  # MixUp nhẹ
        
        # ── Save & Output ──
        save_period=10,             # Lưu checkpoint mỗi 10 epochs
        project=str(OUTPUT_DIR),
        name="yolo11s_vn_traffic",
        exist_ok=True,              # Ghi đè nếu đã có
        plots=True,                 # Tạo biểu đồ
        verbose=True,
    )

elapsed = time.time() - start_time
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)

print("\n" + "=" * 50)
print(f"✅ TRAINING HOÀN TẤT! ({hours}h {minutes}m)")
print("=" * 50)

---
## 5. Đánh giá kết quả Fine-tuned model

In [ ]:
from pathlib import Path
import glob
from ultralytics import YOLO

DATA_YAML = globals().get('DATA_YAML', Path("/kaggle/working/data.yaml"))

run_dir = None

# Cách 1: Lấy trực tiếp từ đối tượng results/model vừa train xong ở Cell 4 (nếu cùng session)
try:
    if ('results' in globals() or 'results' in locals()) and hasattr(results, 'save_dir'):
        run_dir = Path(results.save_dir)
    elif ('model' in globals() or 'model' in locals()) and hasattr(model, 'trainer') and hasattr(model.trainer, 'save_dir'):
        run_dir = Path(model.trainer.save_dir)
except Exception:
    pass

# Cách 2: Tự động quét tìm file best.pt trong toàn bộ /kaggle/working
if run_dir is None or not (run_dir / "weights" / "best.pt").exists():
    found_weights = list(Path("/kaggle/working").rglob("weights/best.pt"))
    if found_weights:
        # Lấy checkpoint mới nhất
        run_dir = sorted(found_weights, key=lambda p: p.stat().st_mtime)[-1].parent.parent

if run_dir is None or not (run_dir / "weights" / "best.pt").exists():
    raise FileNotFoundError(
        "❌ Không tìm thấy weights/best.pt!\n"
        "Nguyên nhân: Cell 4 chưa chạy xong, bị lỗi ngắt giữa chừng, hoặc session Kaggle đã bị reset."
    )

best_pt = run_dir / "weights" / "best.pt"
print(f"📂 Thư mục training: {run_dir}")
print(f"📦 Best weights:     {best_pt}")
print(f"   Kích thước:       {best_pt.stat().st_size / (1024*1024):.1f} MB")

# Đánh giá trên tập validation
model_eval = YOLO(str(best_pt))
metrics = model_eval.val(data=str(DATA_YAML))

print("\n" + "=" * 60)
print("📊 KẾT QUẢ FINE-TUNED MODEL")
print("=" * 60)
print(f"   mAP@0.5:        {metrics.box.map50:.4f}")
print(f"   mAP@0.5:0.95:   {metrics.box.map:.4f}")
print(f"   Precision:       {metrics.box.mp:.4f}")
print(f"   Recall:          {metrics.box.mr:.4f}")

# Kết quả từng class
class_names = ['car', 'motorcycle', 'truck', 'bus', 'person', 'bicycle']
print(f"\n{'Class':<15} {'mAP@0.5':>10} {'mAP@0.5:0.95':>15}")
print("-" * 42)
for i, name in enumerate(class_names):
    try:
        ap50 = metrics.box.ap50[i]
        ap = metrics.box.ap[i]
        marker = " ⭐" if name in ['motorcycle', 'person'] else ""
        print(f"   {name:<12} {ap50:>10.4f} {ap:>15.4f}{marker}")
    except (IndexError, Exception):
        print(f"   {name:<12} {'N/A':>10} {'N/A':>15}")
print("=" * 60)
print("⭐ = Class quan trọng nhất cho giao thông VN")


---
## 6. So sánh với model gốc COCO (Trước vs Sau fine-tune)

In [ ]:
from pathlib import Path
import glob
from ultralytics import YOLO

# Khởi tạo fallback biến môi trường nếu kernel vừa restart
DATA_YAML = globals().get('DATA_YAML', Path("/kaggle/working/data.yaml"))

if 'class_names' not in globals():
    class_names = ['car', 'motorcycle', 'truck', 'bus', 'person', 'bicycle']

# Tự động nạp hoặc đánh giá lại fine-tuned model nếu biến 'metrics' chưa có trong RAM
if 'metrics' not in globals():
    print("⚠️ Biến 'metrics' chưa có trong RAM (do chưa chạy Mục 5 hoặc kernel đã restart).")
    print("🔄 Đang tự động tìm weights và đánh giá fine-tuned model...")
    
    if 'best_pt' not in globals() or not Path(str(best_pt)).exists():
        found_weights = list(Path("/kaggle/working").rglob("weights/best.pt"))
        if found_weights:
            best_pt = sorted(found_weights, key=lambda p: p.stat().st_mtime)[-1]
        else:
            best_pt = None

    if best_pt and Path(str(best_pt)).exists():
        print(f"📦 Đang đánh giá model từ: {best_pt}")
        model_eval = YOLO(str(best_pt))
        metrics = model_eval.val(data=str(DATA_YAML))
        print("✅ Đã tính toán xong 'metrics' cho fine-tuned model!\n")
    else:
        raise FileNotFoundError(
            "❌ Không tìm thấy weights/best.pt!\n"
            "Nguyên nhân: Cell 4 chưa chạy xong, bị lỗi ngắt giữa chừng, hoặc session Kaggle đã bị reset.\n"
            "Hãy đảm bảo bạn đã hoàn thành bước training trước khi so sánh."
        )

print("🔍 Đánh giá model GỐC (COCO pretrained) trên cùng dataset...")
print("   (Để so sánh hiệu quả fine-tuning)\n")

model_original = YOLO("yolo11s.pt")
metrics_orig = model_original.val(data=str(DATA_YAML))

# Bảng so sánh
print("\n" + "=" * 70)
print("📊 SO SÁNH: COCO GỐC vs FINE-TUNED (VN Traffic)")
print("=" * 70)
print(f"{'Metric':<20} {'COCO gốc':>12} {'Fine-tuned':>12} {'Thay đổi':>12}")
print("-" * 58)

comparisons = [
    ('mAP@0.5', metrics_orig.box.map50, metrics.box.map50),
    ('mAP@0.5:0.95', metrics_orig.box.map, metrics.box.map),
    ('Precision', metrics_orig.box.mp, metrics.box.mp),
    ('Recall', metrics_orig.box.mr, metrics.box.mr),
]

for name, orig, fine in comparisons:
    diff = fine - orig
    arrow = "🟢 +" if diff > 0 else "🔴 "
    print(f"   {name:<17} {orig:>12.4f} {fine:>12.4f} {arrow}{diff*100:.2f}%")

# So sánh từng class
print(f"\n{'Class':<15} {'COCO gốc':>12} {'Fine-tuned':>12} {'Thay đổi':>12}")
print("-" * 53)
for i, cls_name in enumerate(class_names):
    try:
        orig_ap = metrics_orig.box.ap50[i]
        fine_ap = metrics.box.ap50[i]
        diff = fine_ap - orig_ap
        arrow = "🟢 +" if diff > 0 else "🔴 "
        marker = " ⭐" if cls_name in ['motorcycle', 'person'] else ""
        print(f"   {cls_name:<12} {orig_ap:>12.4f} {fine_ap:>12.4f} {arrow}{diff*100:.2f}%{marker}")
    except (IndexError, Exception):
        print(f"   {cls_name:<12} {'N/A':>12} {'N/A':>12}")

print("=" * 70)
print("⭐ = Target class (mục tiêu cải thiện ≥ +8%)")


---
## 7. 💾 Lưu kết quả & Chuẩn bị download

### Cách lấy file weights về:
1. Click **"Save Version"** (góc trên phải) → **"Save & Run All (Commit)"**
2. Đợi notebook chạy xong
3. Vào tab **"Output"** → Download folder `model_output/`
4. Upload `best.pt` và `last.pt` lên Google Drive tại:
   `MyDrive/accident_detection/checkpoints/yolo11s_vn_traffic/`

In [ ]:
import shutil
from pathlib import Path
import glob

# Tạo folder output gọn gàng
SAVE_DIR = Path("/kaggle/working/model_output")
SAVE_DIR.mkdir(exist_ok=True)

# Đảm bảo run_dir tồn tại ngay cả khi kernel restart
if 'run_dir' not in globals() or run_dir is None or not Path(str(run_dir)).exists():
    OUTPUT_DIR = globals().get('OUTPUT_DIR', Path("/kaggle/working/runs"))
    runs = sorted(glob.glob(f"{OUTPUT_DIR}/yolo11s_vn_traffic*"))
    run_dir = Path(runs[-1]) if runs else None

if run_dir is None or not run_dir.exists():
    raise FileNotFoundError("Không tìm thấy thư mục training!")

print("💾 Đang chuẩn bị file để download...\n")

# Copy weights
weights_dir = run_dir / "weights"
for f_name in ["best.pt", "last.pt"]:
    src = weights_dir / f_name
    if src.exists():
        shutil.copy(src, SAVE_DIR / f_name)
        size_mb = src.stat().st_size / (1024 * 1024)
        print(f"   ✅ {f_name} ({size_mb:.1f} MB)")
    else:
        print(f"   ⚠️ {f_name} không tìm thấy")

# Copy biểu đồ training và metrics
copied_extras = []
for pattern in ["*.png", "*.jpg", "*.csv"]:
    for f in run_dir.glob(pattern):
        shutil.copy(f, SAVE_DIR / f.name)
        copied_extras.append(f.name)

if copied_extras:
    print(f"\n   📊 Biểu đồ & metrics: {', '.join(copied_extras[:5])}")
    if len(copied_extras) > 5:
        print(f"      ... và {len(copied_extras) - 5} file khác")

# Hiển thị hướng dẫn
print(f"\n{'=' * 60}")
print(f"📦 TẤT CẢ FILE ĐÃ LƯU TẠI: {SAVE_DIR}")
print(f"{'=' * 60}")
print()
print("📥 CÁCH TẢI VỀ MÁY:")
print("   1. Click 'Save Version' (góc phải trên)")
print("   2. Chọn 'Save & Run All (Commit)'")
print("   3. Đợi chạy xong → Vào tab 'Output'")
print("   4. Download folder 'model_output/'")
print()
print("📤 SAU KHI TẢI VỀ, UPLOAD LÊN GOOGLE DRIVE:")
print("   MyDrive/accident_detection/checkpoints/yolo11s_vn_traffic/")
print("   ├── best.pt     ← Model tốt nhất")
print("   └── last.pt     ← Checkpoint cuối (để resume nếu cần)")
print(f"{'=' * 60}")


---
## 8. (Tùy chọn) Thử inference trên ảnh mẫu

In [ ]:
from IPython.display import Image, display
import random
import glob
from pathlib import Path
from ultralytics import YOLO

WORK_DIR = globals().get('WORK_DIR', Path("/kaggle/working"))
DATA_DIR = globals().get('DATA_DIR', WORK_DIR / "dataset")
OUTPUT_DIR = globals().get('OUTPUT_DIR', WORK_DIR / "runs")

# Đảm bảo best_pt tồn tại
if 'best_pt' not in globals() or not Path(str(best_pt)).exists():
    runs = sorted(glob.glob(f"{OUTPUT_DIR}/yolo11s_vn_traffic*"))
    run_dir = Path(runs[-1]) if runs else None
    best_pt = (run_dir / "weights" / "best.pt") if run_dir else None

# Chọn ngẫu nhiên vài ảnh validation để test
val_images = glob.glob(str(DATA_DIR / "images/val/*.jpg")) +              glob.glob(str(DATA_DIR / "images/val/*.png"))

if val_images and best_pt and Path(str(best_pt)).exists():
    sample_imgs = random.sample(val_images, min(4, len(val_images)))
    
    print("🖼️ Inference trên ảnh mẫu từ tập validation:\n")
    model_test = YOLO(str(best_pt))
    results = model_test(sample_imgs, save=True, conf=0.4, 
                         project=str(WORK_DIR / "inference_test"),
                         name="samples", exist_ok=True)
    
    # Hiển thị kết quả
    result_dir = WORK_DIR / "inference_test" / "samples"
    for img_file in sorted(result_dir.glob("*.*"))[:4]:
        print(f"\n📸 {img_file.name}")
        display(Image(filename=str(img_file), width=600))
elif not (best_pt and Path(str(best_pt)).exists()):
    print("⚠️ Không tìm thấy file 'best.pt' để test inference.")
else:
    print("Không tìm thấy ảnh validation để test.")
